In [7]:
import sys
import os
# Get the current directory and parent directory
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
sys.path.insert(0, current_dir)

# Import from the-machine module
import importlib.util
the_machine_path = os.path.join(parent_dir, "the-machine")
sys.path.insert(0, the_machine_path)

# Import all from the-machine and custom_matroid_functions
from HM_module import *
from custom_matroid_functions import *

In [8]:
P = LaurentPolynomialRing(ZZ, 'x')
x = P.gen()
k, n = 5, 6
#M = matroids.CompleteGraphic(n)
M = matroids.Uniform(k, n)
#M = matroids.catalog.Fano()
L = M.lattice_of_flats()
r = M.rank()
rank_func = lambda x : M.rank(x)

#e = next(x for x in M.groundset() if x  not in M.coloops())
e = list(M.groundset())[0]
print(M.groundset())
print("deleting ", e)
mdele = M.delete(e)
Eq_M = create_deformed_algebra(M, base_ring=P)
Eq_mdele = create_deformed_algebra(mdele, base_ring=P)
E_mdele = Eq_mdele.natural()

E = Eq_M.natural()
E_basis = E.basis()
E_mdele_basis = E_mdele.basis()

C = Eq_M.C()
C_dele = Eq_mdele.C()
C_basis = C.basis()
C_dele_basis = C_dele.basis()

Z = Eq_M.zeta()
Z_dele = Eq_mdele.zeta()
Z_basis = Z.basis()

flats_keys = list(E_basis.keys())

chow = chow_polynomial
chibar = reduced_characteristic_polynomial
kl_poly = lambda M: M.lattice_of_flats().kazhdan_lusztig_polynomial()(x)

frozenset({0, 1, 2, 3, 4, 5})
deleting  0


In [9]:
def flat_division(M, e):
    L = M.lattice_of_flats()
    flat_div = {'S':set(), 'T':set(), 'A':set(), 'B':set()}
    for F in L:
        if e in F:
            if F - {e} in L:
                flat_div['T'].add(F)
            else:
                flat_div['A'].add(F)
        else:
            if F.union({e}) in L:
                flat_div['S'].add(F)
            else:
                flat_div['B'].add(F)
    return flat_div

flat_div = flat_division(M, e)
#flat_div['S'].remove(frozenset())

def convolve(f, g, M):
    E = M.groundset()
    flats = list(M.lattice_of_flats())
    sum = P(0)
    for F in flats:
        sum += f(M.delete(E - F))(x) * g(M.contract(F))(x)
    return sum

def rev(f):
    return lambda M: f(M)(x**-1) * x**M.rank()

def rightAugpoly(M):
    return convolve(chow, rev(kl_poly), M)

print(rightAugpoly(M))

1 + 66*x + 376*x^2 + 376*x^3 + 66*x^4 + x^5


In [10]:
print("rank of M: ", r)
print("chow polynomial of M: ", chow_polynomial(M))
print(reduced_characteristic_polynomial(M))
for F in flat_div['S']:
    print(F)

rank of M:  5
chow polynomial of M:  x^4 + 51*x^3 + 161*x^2 + 51*x + 1
x^4 - 5*x^3 + 10*x^2 - 10*x + 5
frozenset({1, 3, 5})
frozenset({1, 4})
frozenset({2, 3, 4})
frozenset({2, 3})
frozenset({4, 5})
frozenset({1})
frozenset({3, 4})
frozenset({2, 4})
frozenset({2, 3, 5})
frozenset({3, 5})
frozenset({3, 4, 5})
frozenset({1, 2, 3})
frozenset({1, 3})
frozenset({1, 3, 4})
frozenset({1, 2, 5})
frozenset({1, 5})
frozenset({1, 4, 5})
frozenset({5})
frozenset({4})
frozenset({2})
frozenset({1, 2, 4})
frozenset({1, 2})
frozenset({2, 4, 5})
frozenset({2, 5})
frozenset({3})
frozenset()


In [11]:


def deleconj(M, e, f):
    mdele = M.delete(e)
    og_matroid_poly = f(M)
    deleted_matroid_poly = f(mdele)
    sum2 = 0
    print("f(M): ", og_matroid_poly)
    for F in flat_div['S']:
        poly1 = chow_polynomial(M.contract(F.union({e})))
        poly2 = f(M.delete(M.groundset() - F))
        term = x * poly1 * poly2
        sum2 += term
        print(F)
        print("--------------------------------")
        print("poly1: ", poly1)
        print("poly2: ", poly2)
        print("term: ", term)
        print("--------------------------------")
        print("********************************")
    print("sum2: ", sum2)
    return P(deleted_matroid_poly + sum2)

deleconj(M, e, rightAugpoly)


f(M):  1 + 66*x + 376*x^2 + 376*x^3 + 66*x^4 + x^5
frozenset({1, 3, 5})
--------------------------------
poly1:  1
poly2:  1 + 7*x + 7*x^2 + x^3
term:  x + 7*x^2 + 7*x^3 + x^4
--------------------------------
********************************
frozenset({1, 4})
--------------------------------
poly1:  x + 1
poly2:  1 + 3*x + x^2
term:  x + 4*x^2 + 4*x^3 + x^4
--------------------------------
********************************
frozenset({2, 3, 4})
--------------------------------
poly1:  1
poly2:  1 + 7*x + 7*x^2 + x^3
term:  x + 7*x^2 + 7*x^3 + x^4
--------------------------------
********************************
frozenset({2, 3})
--------------------------------
poly1:  x + 1
poly2:  1 + 3*x + x^2
term:  x + 4*x^2 + 4*x^3 + x^4
--------------------------------
********************************
frozenset({4, 5})
--------------------------------
poly1:  x + 1
poly2:  1 + 3*x + x^2
term:  x + 4*x^2 + 4*x^3 + x^4
--------------------------------
********************************
frozenset({1})


1 + 57*x + 302*x^2 + 302*x^3 + 57*x^4 + x^5

In [12]:
print(rightAugpoly(M.delete(e)))
print(rightAugpoly(M.contract(e)))

1 + 31*x + 131*x^2 + 131*x^3 + 31*x^4 + x^5
1 + 31*x + 81*x^2 + 31*x^3 + x^4
